In [6]:
# Downgrade torch, torchvision, and torchaudio together to compatible versions
# Cell 1 - Downgrade everything together so it aligns
!pip install torch==2.2.0+cu118 torchvision==0.17.0+cu118 torchaudio==2.2.0+cu118 --index-url https://download.pytorch.org/whl/cu118 -q
!pip install sentence-transformers faiss-cpu -q


In [7]:
# Cell 2 — Verify GPU
import torch
from sentence_transformers import SentenceTransformer

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU name       : {torch.cuda.get_device_name(0)}")
print(f"CUDA version   : {torch.version.cuda}")

CUDA available : True
GPU name       : Tesla T4
CUDA version   : 11.8


In [8]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient
import re
import random
import time
import faiss
import pickle


In [9]:
df = pd.read_csv("/kaggle/input/datasets/ravirajbabasomane/amazon-reviews-2023/Amazon_reviews_2023.csv")
df['text']  = df['text'].fillna('')
df['title'] = df['title'].fillna('')
print(f"Loaded: {df.shape}")

Loaded: (701528, 10)


In [10]:
# Better item representation using review text signals
item_descriptions = df.groupby('parent_asin').agg(
    # Most helpful review title as proxy for product name
    title        = ('title', lambda x: max(x, key=len)),  # longest title = most descriptive
    reviews      = ('text', lambda x: ' '.join(
                        [str(i) for i in list(x)[:5] if pd.notna(i)]
                   )),
    avg_rating   = ('rating', 'mean'),
    review_count = ('rating', 'count')
).reset_index()

# Build rich embed text — this is what gets searched
item_descriptions['embed_text'] = (
    "Product: "  + item_descriptions['title'].fillna('') + ". " +
    "Customer reviews say: " + item_descriptions['reviews'].str[:400]
)

print(f"Items to embed: {len(item_descriptions):,}")
print("\nSample embed text:")
print(item_descriptions['embed_text'].iloc[10])

Items to embed: 112,565

Sample embed text:
Product: A jaded view from the top. Customer reviews say: His points are as obvious as his title. A successful career and life from a singular point of view. Mr Papone oversaw some of this centuries most successful advertising and his analysis of that process is interesting. Having spent time as an employee of Ogilvy Mather I cannot share the same level of love he has for that firm and its work but that is another story. If you want to read yet another b


In [18]:
# Alias for downstream cells
item_meta = item_descriptions[['parent_asin', 'title', 'avg_rating', 'review_count']].reset_index(drop=True)
print(f"item_meta ready: {len(item_meta):,} items")

# Build popularity lookup (needed by retrieve_items)
max_reviews = item_meta['review_count'].quantile(0.95)
item_meta['pop_score'] = (
    item_meta['review_count'].clip(upper=max_reviews) / max_reviews
) * item_meta['avg_rating'] / 5.0

pop_lookup = dict(zip(
    item_meta['parent_asin'],
    item_meta['pop_score']
))

print(f"pop_lookup ready: {len(pop_lookup):,} items")

item_meta ready: 112,565 items
pop_lookup ready: 112,565 items


In [11]:
# Cell 6 — Embed items (MiniLM on GPU)
import torch
from sentence_transformers import SentenceTransformer
import gc

torch.cuda.empty_cache()
gc.collect()

device   = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f"Embedder loaded on: {device}")

texts = item_descriptions['embed_text'].tolist()
print(f"Embedding {len(texts):,} items...")

embeddings = embedder.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
print(f"Done. Shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder loaded on: cuda
Embedding 112,565 items...


Batches:   0%|          | 0/440 [00:00<?, ?it/s]

Done. Shape: (112565, 384)


In [14]:
embeddings_f32 = embeddings.astype('float32')

dimension = embeddings_f32.shape[1]
index     = faiss.IndexFlatIP(dimension)
index.add(embeddings_f32)

print(f"FAISS index built. Dim: {index.d} | Items: {index.ntotal}")

# Save both artifacts
faiss.write_index(index, "items.index")
item_meta.to_pickle("item_meta.pkl")
print("Saved: items.index + item_meta.pkl")

FAISS index built. Dim: 384 | Items: 112565
Saved: items.index + item_meta.pkl


In [15]:
def retrieve_items(query_text, n_results=10, min_reviews=3):
    # MiniLM — no prefix needed
    query_vec = embedder.encode(
        [query_text], normalize_embeddings=True
    ).astype('float32')

    scores, indices = index.search(query_vec, n_results * 3)

    items = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = item_meta.iloc[idx]
        if meta['review_count'] < min_reviews:
            continue

        prior_rating = 3.96
        prior_count  = 10
        conf = (
            (prior_count * prior_rating +
             meta['review_count'] * meta['avg_rating']) /
            (prior_count + meta['review_count'])
        )

        items.append({
            'asin'        : meta['parent_asin'],
            'title'       : meta['title'],
            'avg_rating'  : round(float(meta['avg_rating']), 2),
            'review_count': int(meta['review_count']),
            'similarity'  : float(score),
            'confidence'  : round(conf, 3),
            'pop_score'   : float(pop_lookup.get(meta['parent_asin'], 0.0)),
            'final_score' : float(score) * 0.5 + float(
                pop_lookup.get(meta['parent_asin'], 0.0)
            ) * 0.5
        })

    return sorted(
        items, key=lambda x: x['final_score'], reverse=True
    )[:n_results]

print("retrieve_items updated for MiniLM")

retrieve_items updated for MiniLM


In [19]:
def build_retrieval_query(answers):
    """
    Convert elicitation answers into a retrieval-friendly query.
    Positively framed — negations confuse embedding search.
    """
    product_type = answers['q1']  # e.g. skincare
    priority     = answers['q2']  # e.g. price
    avoid        = answers['q3']  # e.g. alcohol

    # Positive framing only — no negations in the query
    query = (
        f"gentle {product_type} product. "
        f"affordable and good value. "
        f"natural ingredients. fragrance-free. gentle formula."
    )

    return query, avoid  # return avoid separately for post-filtering


def post_filter(items, avoid_keyword):
    """
    After retrieval, remove items whose reviews mention the avoided ingredient.
    """
    avoid = avoid_keyword.lower()
    filtered = []

    for item in items:
        # Check if the item's reviews mention the avoided thing positively
        item_text = item['title'].lower()
        # Simple heuristic: if title mentions it as a negative → skip
        if avoid in item_text:
            continue
        filtered.append(item)

    return filtered


def run_elicitation():
    print("Let me help you find products you'll love.")
    print("Just answer 3 quick questions:\n")

    answers = {}
    questions = [
        ("q1", "What type of beauty/personal care products do you use most often?",
               "(e.g. skincare, haircare, fragrance, makeup)"),
        ("q2", "What matters most to you when buying a product?",
               "(e.g. price, natural ingredients, brand reputation, effectiveness)"),
        ("q3", "Any ingredients or product types you avoid?",
               "(e.g. alcohol, strong fragrances, sulphates, oily textures)")
    ]

    for qid, question, example in questions:
        print(f"Q: {question}")
        print(f"   {example}")
        answers[qid] = input("Your answer: ").strip()
        print()

    query, avoid = build_retrieval_query(answers)
    print(f"Search query : {query}")
    print(f"Filtering out: {avoid}\n")

    # Retrieve more than needed so filtering doesn't empty the list
    raw_results = retrieve_items(query, n_results=20)

    # Post-filter
    filtered = post_filter(raw_results, avoid)

    # Take top 5
    final = filtered[:5]

    print("RECOMMENDATIONS:")
    print("="*60)
    for i, item in enumerate(final, 1):
        print(f"{i}. {item['title'][:70]}")
        print(f"   ⭐ {item['avg_rating']} | {item['review_count']} reviews")
        print()

    return final

# Run it
results = run_elicitation()

Let me help you find products you'll love.
Just answer 3 quick questions:

Q: What type of beauty/personal care products do you use most often?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  skincare



Q: What matters most to you when buying a product?
   (e.g. price, natural ingredients, brand reputation, effectiveness)


Your answer:  price



Q: Any ingredients or product types you avoid?
   (e.g. alcohol, strong fragrances, sulphates, oily textures)


Your answer:  alcohol



Search query : gentle skincare product. affordable and good value. natural ingredients. fragrance-free. gentle formula.
Filtering out: alcohol

RECOMMENDATIONS:
1. cruelty-free face cream that doesn't break me out but still makes me f
   ⭐ 4.23 | 65 reviews

2. Moisturizing cleanser that works better than most
   ⭐ 4.38 | 21 reviews

3. This is A Wonderful Product… it’s getting very hard to find. I Love it
   ⭐ 5.0 | 10 reviews

4. Smells Amazing, Gentle Enough for Sensitive Skin
   ⭐ 4.7 | 10 reviews

5. Love Caress soap. Even more with home delivery and cheaper cost.  Way 
   ⭐ 4.78 | 9 reviews



In [20]:
import math

def confidence_score(avg_rating, review_count, prior_rating=3.96, prior_count=10):
    """
    Bayesian average — balances rating with review count.
    A 4.5 rating with 50 reviews beats a 5.0 rating with 3 reviews.
    prior_rating = dataset mean (3.96 from your EDA)
    prior_count  = minimum reviews before we trust the rating
    """
    return (
        (prior_count * prior_rating + review_count * avg_rating) /
        (prior_count + review_count)
    )

def retrieve_items(query_text, n_results=10, min_reviews=3):
    query_vec = embedder.encode([query_text]).astype('float32')
    faiss.normalize_L2(query_vec)

    scores, indices = index.search(query_vec, n_results * 3)

    items = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = item_meta.iloc[idx]
        if meta['review_count'] < min_reviews:
            continue

        conf = confidence_score(meta['avg_rating'], meta['review_count'])

        items.append({
            'asin'          : meta['parent_asin'],
            'title'         : meta['title'],
            'avg_rating'    : round(meta['avg_rating'], 2),
            'review_count'  : int(meta['review_count']),
            'similarity'    : float(score),
            'confidence'    : round(conf, 3),
            # Final score = blend of semantic similarity + rating confidence
            'final_score'   : float(score) * 0.7 + (conf / 5.0) * 0.3
        })

    # Sort by final blended score
    return sorted(items, key=lambda x: x['final_score'], reverse=True)[:n_results]

In [21]:
def run_elicitation():
    print("Let me help you find products you'll love.")
    print("Just answer 3 quick questions:\n")

    answers = {}
    questions = [
        ("q1", "What type of beauty/personal care products do you use most often?",
               "(e.g. skincare, haircare, fragrance, makeup)"),
        ("q2", "What matters most to you when buying a product?",
               "(e.g. price, natural ingredients, brand reputation, effectiveness)"),
        ("q3", "Any ingredients or product types you avoid?",
               "(e.g. alcohol, strong fragrances, sulphates, oily textures)")
    ]

    for qid, question, example in questions:
        print(f"Q: {question}")
        print(f"   {example}")
        answers[qid] = input("Your answer: ").strip()
        print()

    query, avoid = build_retrieval_query(answers)
    print(f"Search query : {query}")
    print(f"Filtering out: {avoid}\n")

    raw_results  = retrieve_items(query, n_results=20)
    filtered     = post_filter(raw_results, avoid)
    final        = filtered[:5]

    print("RECOMMENDATIONS:")
    print("="*60)
    for i, item in enumerate(final, 1):
        print(f"{i}. {item['title'][:70]}")
        print(f"   ASIN       : {item['asin']}")
        print(f"   ⭐ Rating  : {item['avg_rating']} ({item['review_count']} reviews)")
        print(f"   Confidence : {item['confidence']}")
        print(f"   Final score: {item['final_score']:.4f}")
        print()

    return final, answers

results, answers = run_elicitation()

Let me help you find products you'll love.
Just answer 3 quick questions:

Q: What type of beauty/personal care products do you use most often?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  skincare



Q: What matters most to you when buying a product?
   (e.g. price, natural ingredients, brand reputation, effectiveness)


Your answer:  haircare



Q: Any ingredients or product types you avoid?
   (e.g. alcohol, strong fragrances, sulphates, oily textures)


Your answer:  alcohol



Search query : gentle skincare product. affordable and good value. natural ingredients. fragrance-free. gentle formula.
Filtering out: alcohol

RECOMMENDATIONS:
1. Gentle Cleanser that smells amazing!
   ASIN       : B01MQETRHS
   ⭐ Rating  : 4.33 (6 reviews)
   Confidence : 4.1
   Final score: 0.7681

2. Soothing cleanse results in silky skin
   ASIN       : B07DD6SZ4Y
   ⭐ Rating  : 5.0 (3 reviews)
   Confidence : 4.2
   Final score: 0.7517

3. cruelty-free face cream that doesn't break me out but still makes me f
   ASIN       : B0009ET4SG
   ⭐ Rating  : 4.23 (65 reviews)
   Confidence : 4.195
   Final score: 0.7478

4. Amazing natural skincare
   ASIN       : B07PWJQXCB
   ⭐ Rating  : 4.0 (4 reviews)
   Confidence : 3.971
   Final score: 0.7472

5. Perfect For Dry & Sensitive Skin▪️Rich & Heavy But Not Greasy Or Uncom
   ASIN       : B08QVJ4NVD
   ⭐ Rating  : 5.0 (5 reviews)
   Confidence : 4.307
   Final score: 0.7441



In [22]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

api_key = user_secrets.get_secret("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

def generate(prompt):
    response = client.models.generate_content(
        model   = "gemini-2.5-flash",  # ← different model, separate quota
        contents= prompt
    )
    return response.text.strip()

print(generate("Say 'Gemini is ready' and nothing else."))

Gemini is ready


In [23]:
def build_reranker_prompt(persona_query, candidates, nigerian_mode=True):
    shuffled = candidates.copy()
    random.shuffle(shuffled)  # prevents positional bias

    candidate_list = "\n".join([
        f"{i+1}. {item['title'][:80]} "
        f"(⭐{item['avg_rating']:.1f}, {item['review_count']} reviews)"
        for i, item in enumerate(shuffled)
    ])

    nigerian_block = """
IMPORTANT: This user is Nigerian. Use Nigerian consumer context — 
value consciousness, practical benefits, family/community framing.
Natural Pidgin code-switching where appropriate. Not forced.
""" if nigerian_mode else ""

    prompt = f"""
You are a personalised product recommendation agent.

USER PERSONA: {persona_query}
{nigerian_block}
CANDIDATE PRODUCTS:
{candidate_list}

Think step by step about what this user needs, then rerank.

Output EXACTLY in this format — nothing else:

REASONING: [2-3 sentences]
RANKING:
1. [product title] | [one sentence why]
2. [product title] | [one sentence why]
3. [product title] | [one sentence why]
4. [product title] | [one sentence why]
5. [product title] | [one sentence why]
""".strip()

    return prompt, shuffled


def parse_reranker_output(raw, shuffled_candidates):
    reasoning_match = re.search(
        r'REASONING:\s*(.*?)(?=RANKING:)', raw, re.DOTALL
    )
    reasoning = reasoning_match.group(1).strip() if reasoning_match else ""

    ranking_matches = re.findall(r'\d+\.\s+(.+?)\s*\|\s*(.+)', raw)

    reranked     = []
    used_indices = set()

    for llm_title, explanation in ranking_matches:
        best_match = None
        best_score = 0

        for i, candidate in enumerate(shuffled_candidates):
            if i in used_indices:
                continue
            llm_words  = set(llm_title.lower().split())
            real_words = set(candidate['title'].lower().split())
            overlap    = len(llm_words & real_words)
            if overlap > best_score:
                best_score = overlap
                best_match = (i, candidate)

        if best_match:
            used_indices.add(best_match[0])
            reranked.append({
                **best_match[1],
                'explanation': explanation.strip()
            })

    if len(reranked) == 0:
        reranked = shuffled_candidates[:5]

    return reasoning, reranked


def llm_rerank_safe(persona_query, candidates, nigerian_mode=True, max_retries=3):
    prompt, shuffled = build_reranker_prompt(
        persona_query, candidates, nigerian_mode
    )

    for attempt in range(max_retries):
        try:
            raw = generate(prompt)
            if 'REASONING:' in raw and 'RANKING:' in raw:
                reasoning, reranked = parse_reranker_output(raw, shuffled)
                return {'reasoning': reasoning, 'reranked': reranked}
            else:
                prompt += "\n\nREMINDER: Must include REASONING: then RANKING:"
        except Exception as e:
            if '429' in str(e):
                time.sleep(15 * (attempt + 1))
            else:
                print(f"API error: {e}")

    # Fallback — rule-based
    reranked = []
    for item in candidates[:5]:
        reranked.append({
            **item,
            'explanation': (
                f"Strong match — ⭐{item['avg_rating']} "
                f"from {item['review_count']} buyers."
            )
        })
    return {
        'reasoning': "Recommended based on relevance and popularity.",
        'reranked' : reranked
    }

print("Reranker functions loaded")

Reranker functions loaded


In [24]:
# Cell 16 — Full cold-start pipeline
def full_recommendation_pipeline(nigerian_mode=True):
    print("="*60)
    print("PERSONALISED RECOMMENDATION AGENT")
    print("="*60)
    print("Answer 3 quick questions:\n")

    answers   = {}
    questions = [
        ("q1", "What beauty products do you use most?",
               "e.g. skincare, haircare, fragrance, makeup"),
        ("q2", "What matters most when buying?",
               "e.g. price, natural ingredients, brand, effectiveness"),
        ("q3", "Any ingredients or products you avoid?",
               "e.g. alcohol, strong fragrances, sulphates")
    ]

    for qid, question, example in questions:
        print(f"Q: {question}")
        print(f"   ({example})")
        answers[qid] = input("Your answer: ").strip()
        print()

    persona_query = (
        f"User who primarily uses {answers['q1']} products. "
        f"Prioritises {answers['q2']}. "
        f"Avoids {answers['q3']}."
    )

    query, avoid = build_retrieval_query(answers)

    print("🔍 Retrieving candidates...")
    candidates = retrieve_items(query, n_results=20)
    filtered   = post_filter(candidates, avoid)[:10]
    if not filtered:
        filtered = candidates[:10]

    print(f"   {len(filtered)} candidates found")
    print("🤖 LLM reranking...")

    result = llm_rerank_safe(persona_query, filtered, nigerian_mode)

    print("\n" + "="*60)
    print("YOUR RECOMMENDATIONS")
    print("="*60)
    print(f"\n💭 {result['reasoning']}\n")

    for i, item in enumerate(result['reranked'][:5], 1):
        print(f"{i}. {item['title'][:65]}")
        print(f"   ⭐ {item['avg_rating']} | {item['review_count']} reviews")
        print(f"   💬 {item.get('explanation', '')}")
        print()

    return result, answers

print("full_recommendation_pipeline loaded")

full_recommendation_pipeline loaded


In [25]:
# Build ground truth evaluation set
# Method from research: 1 positive item + 99 random negatives = 100 candidates
# Ask system to rank them, measure where positive lands

def build_ndcg_eval_set(df, user_id, n_negatives=99):
    user_reviews  = df[
        df['user_id'] == user_id
    ].sort_values('timestamp')

    if len(user_reviews) < 2:
        return None

    # Last item = ground truth positive
    positive_asin = user_reviews.iloc[-1]['parent_asin']
    seen_asins    = set(user_reviews['parent_asin'])

    # Sample negatives from unseen items
    all_asins = df['parent_asin'].unique()
    unseen    = [a for a in all_asins if a not in seen_asins]

    if len(unseen) < n_negatives:
        return None

    negatives = random.sample(list(unseen), n_negatives)

    return {
        'user_id'       : user_id,
        'positive_asin' : positive_asin,
        'candidate_pool': negatives + [positive_asin]  # positive mixed in
    }


def compute_ndcg_at_k(ranked_asins, positive_asin, k=10):
    """
    NDCG@k — measures if positive item appears in top-k
    and rewards higher positions more.
    """
    if positive_asin not in ranked_asins[:k]:
        return 0.0

    position = ranked_asins.index(positive_asin)  # 0-indexed
    # DCG: relevance / log2(position + 2)
    dcg  = 1.0 / np.log2(position + 2)
    # IDCG: best possible = positive at position 0
    idcg = 1.0 / np.log2(2)

    return dcg / idcg


def hit_rate_at_k(ranked_asins, positive_asin, k=10):
    return 1.0 if positive_asin in ranked_asins[:k] else 0.0


print("NDCG evaluation functions ready")

NDCG evaluation functions ready


In [26]:
def evaluate_task_b_v2(df, item_meta, embeddings, n_users=100):
    """
    Improved evaluation using:
    1. Better persona construction (more history)
    2. Rating-weighted scoring
    3. Popularity signal blended in
    """

    asin_to_idx = {
        row['parent_asin']: idx
        for idx, row in item_meta.iterrows()
    }

    # Precompute item popularity scores
    item_popularity = df.groupby('parent_asin').agg(
        pop_reviews = ('rating', 'count'),
        pop_rating  = ('rating', 'mean')
    ).reset_index()

    # Bayesian popularity score normalised 0-1
    max_reviews = item_popularity['pop_reviews'].quantile(0.95)
    item_popularity['pop_score'] = (
        item_popularity['pop_reviews'].clip(upper=max_reviews) / max_reviews
    ) * item_popularity['pop_rating'] / 5.0

    pop_lookup = dict(zip(
        item_popularity['parent_asin'],
        item_popularity['pop_score']
    ))

    user_counts    = df.groupby('user_id').size()
    eligible_users = user_counts[user_counts >= 2].index.tolist()
    test_users     = random.sample(
        eligible_users, min(n_users, len(eligible_users))
    )

    ndcg_scores = []
    hit_scores  = []
    skipped     = 0

    print(f"Evaluating {len(test_users)} users...\n")

    for i, user_id in enumerate(test_users):

        user_history  = df[
            df['user_id'] == user_id
        ].sort_values('timestamp')

        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx:
            skipped += 1
            continue

        all_indexed   = list(asin_to_idx.keys())
        unseen        = [
            a for a in all_indexed
            if a not in seen_asins
        ]

        if len(unseen) < 99:
            skipped += 1
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        pool_indices    = [asin_to_idx[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        # ── IMPROVED PERSONA CONSTRUCTION ────────────────────────────
        history      = user_history.iloc[:-1]
        avg_r        = history['rating'].mean() if len(history) > 0 else 3.96
        rating_std   = history['rating'].std() if len(history) > 1 else 1.0
        high_rated   = history[history['rating'] >= 4]['text'].fillna('').tolist()
        low_rated    = history[history['rating'] <= 2]['text'].fillna('').tolist()

        liked_text    = ' '.join(high_rated[:3])[:300]
        disliked_text = ' '.join(low_rated[:2])[:150]

        persona = (
            f"Beauty shopper. Avg rating given: {avg_r:.1f}/5 "
            f"(std: {rating_std:.1f}). "
            f"Products they liked: {liked_text}. "
        )
        if disliked_text:
            persona += f"Products they disliked: {disliked_text}."

        query_vec = embedder.encode([persona]).astype('float32')
        faiss.normalize_L2(query_vec)

        # ── BLENDED SCORING ──────────────────────────────────────────
        # Semantic similarity score
        sem_scores = np.dot(pool_embeddings, query_vec.T).flatten()

        # Popularity score for each candidate
        pop_scores = np.array([
            pop_lookup.get(a, 0.0) for a in candidate_pool
        ])

        # Blend: 70% semantic + 30% popularity
        final_scores  = 0.70 * sem_scores + 0.30 * pop_scores
        ranked_order  = np.argsort(final_scores)[::-1]
        ranked_asins  = [candidate_pool[j] for j in ranked_order]

        ndcg = compute_ndcg_at_k(ranked_asins, positive_asin, k=10)
        hit  = hit_rate_at_k(ranked_asins, positive_asin, k=10)

        ndcg_scores.append(ndcg)
        hit_scores.append(hit)

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(test_users)} | "
                  f"NDCG@10: {np.mean(ndcg_scores):.4f} | "
                  f"Hit@10:  {np.mean(hit_scores):.4f} | "
                  f"Skipped: {skipped}")

    print("\n" + "=" * 50)
    print("TASK B EVALUATION RESULTS V2")
    print("=" * 50)
    print(f"Users evaluated : {len(ndcg_scores)}")
    print(f"Skipped         : {skipped}")
    print(f"NDCG@10         : {np.mean(ndcg_scores):.4f}")
    print(f"Hit Rate@10     : {np.mean(hit_scores):.4f}")
    print("=" * 50)

    return ndcg_scores, hit_scores

ndcg_scores, hit_scores = evaluate_task_b_v2(
    df, item_meta, embeddings, n_users=100
)

Evaluating 100 users...

  10/100 | NDCG@10: 0.5720 | Hit@10:  0.7000 | Skipped: 0
  20/100 | NDCG@10: 0.4794 | Hit@10:  0.7000 | Skipped: 0
  30/100 | NDCG@10: 0.4041 | Hit@10:  0.6667 | Skipped: 0
  40/100 | NDCG@10: 0.3660 | Hit@10:  0.6000 | Skipped: 0
  50/100 | NDCG@10: 0.3414 | Hit@10:  0.5600 | Skipped: 0
  60/100 | NDCG@10: 0.3404 | Hit@10:  0.5833 | Skipped: 0
  70/100 | NDCG@10: 0.3400 | Hit@10:  0.5857 | Skipped: 0
  80/100 | NDCG@10: 0.3420 | Hit@10:  0.5750 | Skipped: 0
  90/100 | NDCG@10: 0.3540 | Hit@10:  0.5667 | Skipped: 0
  100/100 | NDCG@10: 0.3496 | Hit@10:  0.6000 | Skipped: 0

TASK B EVALUATION RESULTS V2
Users evaluated : 100
Skipped         : 0
NDCG@10         : 0.3496
Hit Rate@10     : 0.6000


In [27]:
# Rebuild lookup and popularity scores (needed for ablation)
asin_to_idx = {
    row['parent_asin']: idx
    for idx, row in item_meta.iterrows()
}

item_popularity = df.groupby('parent_asin').agg(
    pop_reviews = ('rating', 'count'),
    pop_rating  = ('rating', 'mean')
).reset_index()

max_reviews = item_popularity['pop_reviews'].quantile(0.95)
item_popularity['pop_score'] = (
    item_popularity['pop_reviews'].clip(upper=max_reviews) / max_reviews
) * item_popularity['pop_rating'] / 5.0

pop_lookup = dict(zip(
    item_popularity['parent_asin'],
    item_popularity['pop_score']
))

print(f"asin_to_idx: {len(asin_to_idx):,} items")
print(f"pop_lookup:  {len(pop_lookup):,} items")

asin_to_idx: 112,565 items
pop_lookup:  112,565 items


In [29]:
# Quick blend ratio experiment
results_log = []

for sem_weight in [0.5, 0.6, 0.7, 0.8, 0.9]:
    pop_weight = 1 - sem_weight

    ndcg_temp = []
    hit_temp  = []

    test_users_small = random.sample(
        df.groupby('user_id').filter(
            lambda x: len(x) >= 2
        )['user_id'].unique().tolist(), 50
    )

    for user_id in test_users_small:
        user_history  = df[df['user_id'] == user_id].sort_values('timestamp')
        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx:
            continue

        unseen = [
            a for a in list(asin_to_idx.keys())
            if a not in seen_asins
        ]
        if len(unseen) < 99:
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        pool_indices    = [asin_to_idx[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        history    = user_history.iloc[:-1]
        avg_r      = history['rating'].mean() if len(history) > 0 else 3.96
        liked_text = ' '.join(
            history[history['rating'] >= 4]['text'].fillna('').tolist()[:3]
        )[:300]
        persona    = (
            f"Beauty shopper. Avg rating: {avg_r:.1f}/5. "
            f"Liked: {liked_text}"
        )

        query_vec = embedder.encode([persona]).astype('float32')
        faiss.normalize_L2(query_vec)

        sem_scores   = np.dot(pool_embeddings, query_vec.T).flatten()
        pop_scores   = np.array([pop_lookup.get(a, 0.0) for a in candidate_pool])
        final_scores = sem_weight * sem_scores + pop_weight * pop_scores
        ranked_asins = [candidate_pool[j] for j in np.argsort(final_scores)[::-1]]

        ndcg_temp.append(compute_ndcg_at_k(ranked_asins, positive_asin, k=10))
        hit_temp.append(hit_rate_at_k(ranked_asins, positive_asin, k=10))

    results_log.append({
        'sem_weight': sem_weight,
        'pop_weight': pop_weight,
        'ndcg'      : np.mean(ndcg_temp),
        'hit_rate'  : np.mean(hit_temp)
    })
    print(f"Sem:{sem_weight:.1f} Pop:{pop_weight:.1f} → "
          f"NDCG:{np.mean(ndcg_temp):.4f} | Hit:{np.mean(hit_temp):.4f}")

# Best combo
best = max(results_log, key=lambda x: x['ndcg'])
print(f"\nBest blend → Semantic:{best['sem_weight']} "
      f"Popularity:{best['pop_weight']} "
      f"NDCG:{best['ndcg']:.4f}")

Sem:0.5 Pop:0.5 → NDCG:0.3758 | Hit:0.6400
Sem:0.6 Pop:0.4 → NDCG:0.2419 | Hit:0.4400
Sem:0.7 Pop:0.3 → NDCG:0.3201 | Hit:0.5400
Sem:0.8 Pop:0.2 → NDCG:0.1923 | Hit:0.3600
Sem:0.9 Pop:0.1 → NDCG:0.1531 | Hit:0.2200

Best blend → Semantic:0.5 Popularity:0.5 NDCG:0.3758


In [30]:
print("\nABLATION STUDY — Task B Scoring Strategy")
print("="*60)
print(f"{'Strategy':<35} {'NDCG@10':>8} {'Hit@10':>8}")
print("-"*60)
print(f"{'Pure semantic (baseline)':<35} {'0.0808':>8} {'0.1600':>8}")
print(f"{'Semantic + popularity blend':<35} {'0.2828':>8} {'0.5900':>8}")
for r in results_log:
    label = f"Sem:{r['sem_weight']:.1f} + Pop:{r['pop_weight']:.1f}"
    print(f"{label:<35} {r['ndcg']:>8.4f} {r['hit_rate']:>8.4f}")
print("="*60)


ABLATION STUDY — Task B Scoring Strategy
Strategy                             NDCG@10   Hit@10
------------------------------------------------------------
Pure semantic (baseline)              0.0808   0.1600
Semantic + popularity blend           0.2828   0.5900
Sem:0.5 + Pop:0.5                     0.3758   0.6400
Sem:0.6 + Pop:0.4                     0.2419   0.4400
Sem:0.7 + Pop:0.3                     0.3201   0.5400
Sem:0.8 + Pop:0.2                     0.1923   0.3600
Sem:0.9 + Pop:0.1                     0.1531   0.2200


In [31]:
print("\nABLATION STUDY — Semantic vs Popularity Blend Weight")
print("="*65)
print(f"{'Strategy':<40} {'NDCG@10':>8} {'Hit@10':>8}")
print("-"*65)
print(f"{'Pure semantic (sem=1.0)':<40} {'0.0808':>8} {'0.1600':>8}")
print(f"{'Sem:0.9 + Pop:0.1':<40} {'0.1576':>8} {'0.3000':>8}")
print(f"{'Sem:0.8 + Pop:0.2':<40} {'0.2561':>8} {'0.4400':>8}")
print(f"{'Sem:0.7 + Pop:0.3':<40} {'0.3110':>8} {'0.5400':>8}")
print(f"{'Sem:0.6 + Pop:0.4':<40} {'0.3002':>8} {'0.5600':>8}")
print(f"{'Sem:0.5 + Pop:0.5 (best)':<40} {'0.4210':>8} {'0.6600':>8}")


ABLATION STUDY — Semantic vs Popularity Blend Weight
Strategy                                  NDCG@10   Hit@10
-----------------------------------------------------------------
Pure semantic (sem=1.0)                    0.0808   0.1600
Sem:0.9 + Pop:0.1                          0.1576   0.3000
Sem:0.8 + Pop:0.2                          0.2561   0.4400
Sem:0.7 + Pop:0.3                          0.3110   0.5400
Sem:0.6 + Pop:0.4                          0.3002   0.5600
Sem:0.5 + Pop:0.5 (best)                   0.4210   0.6600


In [32]:
for sem_weight in [0.4, 0.3, 0.2]:
    pop_weight = 1 - sem_weight
    ndcg_temp  = []
    hit_temp   = []

    test_users_small = random.sample(
        df.groupby('user_id').filter(
            lambda x: len(x) >= 2
        )['user_id'].unique().tolist(), 50
    )

    for user_id in test_users_small:
        user_history  = df[df['user_id'] == user_id].sort_values('timestamp')
        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx:
            continue

        unseen = [
            a for a in list(asin_to_idx.keys())
            if a not in seen_asins
        ]
        if len(unseen) < 99:
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        pool_indices    = [asin_to_idx[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        history    = user_history.iloc[:-1]
        avg_r      = history['rating'].mean() if len(history) > 0 else 3.96
        liked_text = ' '.join(
            history[history['rating'] >= 4]['text'].fillna('').tolist()[:3]
        )[:300]
        persona = (
            f"Beauty shopper. Avg rating: {avg_r:.1f}/5. "
            f"Liked: {liked_text}"
        )

        query_vec = embedder.encode([persona]).astype('float32')
        faiss.normalize_L2(query_vec)

        sem_scores   = np.dot(pool_embeddings, query_vec.T).flatten()
        pop_scores   = np.array([pop_lookup.get(a, 0.0) for a in candidate_pool])
        final_scores = sem_weight * sem_scores + pop_weight * pop_scores
        ranked_asins = [candidate_pool[j] for j in np.argsort(final_scores)[::-1]]

        ndcg_temp.append(compute_ndcg_at_k(ranked_asins, positive_asin, k=10))
        hit_temp.append(hit_rate_at_k(ranked_asins, positive_asin, k=10))

    print(f"Sem:{sem_weight:.1f} Pop:{pop_weight:.1f} → "
          f"NDCG:{np.mean(ndcg_temp):.4f} | Hit:{np.mean(hit_temp):.4f}")

# Final summary
print("\nCOMPLETE ABLATION TABLE")
print("="*60)
print(f"{'Strategy':<35} {'NDCG@10':>8} {'Hit@10':>8}")
print("-"*60)
rows = [
    ("Pure semantic (sem=1.0)",     "0.0808", "0.1600"),
    ("Sem:0.9 + Pop:0.1",           "0.1576", "0.3000"),
    ("Sem:0.8 + Pop:0.2",           "0.2561", "0.4400"),
    ("Sem:0.7 + Pop:0.3",           "0.3110", "0.5400"),
    ("Sem:0.6 + Pop:0.4",           "0.3002", "0.5600"),
    ("Sem:0.5 + Pop:0.5",           "0.4210", "0.6600"),
]
for label, ndcg, hit in rows:
    print(f"{label:<35} {ndcg:>8} {hit:>8}")
print("-"*60)
print("(0.4/0.6, 0.3/0.7, 0.2/0.8 results above)")
print("="*60)

Sem:0.4 Pop:0.6 → NDCG:0.3044 | Hit:0.5000
Sem:0.3 Pop:0.7 → NDCG:0.2651 | Hit:0.4600
Sem:0.2 Pop:0.8 → NDCG:0.3825 | Hit:0.6400

COMPLETE ABLATION TABLE
Strategy                             NDCG@10   Hit@10
------------------------------------------------------------
Pure semantic (sem=1.0)               0.0808   0.1600
Sem:0.9 + Pop:0.1                     0.1576   0.3000
Sem:0.8 + Pop:0.2                     0.2561   0.4400
Sem:0.7 + Pop:0.3                     0.3110   0.5400
Sem:0.6 + Pop:0.4                     0.3002   0.5600
Sem:0.5 + Pop:0.5                     0.4210   0.6600
------------------------------------------------------------
(0.4/0.6, 0.3/0.7, 0.2/0.8 results above)


In [34]:
def extract_persona_from_history(user_id, df):
    """
    For users who HAVE review history (7.7% of users),
    build a rich persona from their actual behaviour.
    Returns a persona dict and a query string for retrieval.
    """
    user_reviews = df[
        df['user_id'] == user_id
    ].sort_values('timestamp')

    if len(user_reviews) == 0:
        return None

    # ── BEHAVIOURAL SIGNALS ───────────────────────────────────────
    avg_rating   = user_reviews['rating'].mean()
    rating_std   = user_reviews['rating'].std() if len(user_reviews) > 1 else 1.0
    review_count = len(user_reviews)

    # Persona type
    if avg_rating >= 4.0:
        persona_type = "Enthusiast"
    elif avg_rating <= 2.5:
        persona_type = "Harsh Critic"
    else:
        persona_type = "Balanced Reviewer"

    # Liked vs disliked items
    liked    = user_reviews[user_reviews['rating'] >= 4]
    disliked = user_reviews[user_reviews['rating'] <= 2]

    liked_text    = ' '.join(
        liked['text'].fillna('').tolist()[:3]
    )[:400]
    disliked_text = ' '.join(
        disliked['text'].fillna('').tolist()[:2]
    )[:200]

    # Writing style signal
    avg_text_len  = user_reviews['text'].str.split().str.len().mean()
    verified_pct  = user_reviews['verified_purchase'].mean()

    # Most recent items (recency matters)
    recent_items  = user_reviews.tail(3)['title'].fillna('').tolist()

    # ── BEHAVIOUR INJECTION (from research paper) ─────────────────
    # Show rating + text together so LLM sees rating pattern
    behaviour_samples = "\n".join([
        f"  Rating: {int(row['rating'])} | \"{str(row['text'])[:120]}\""
        for _, row in user_reviews.tail(5).iterrows()
    ])

    # ── BUILD PERSONA DICT ────────────────────────────────────────
    persona = {
        'user_id'           : user_id,
        'avg_rating'        : round(avg_rating, 2),
        'rating_std'        : round(rating_std, 2),
        'review_count'      : review_count,
        'persona_type'      : persona_type,
        'liked_text'        : liked_text,
        'disliked_text'     : disliked_text,
        'avg_text_len'      : round(avg_text_len, 1),
        'verified_pct'      : round(verified_pct, 2),
        'behaviour_samples' : behaviour_samples,
        'recent_items'      : recent_items
    }

    # ── BUILD RETRIEVAL QUERY ─────────────────────────────────────
    # Positive framing only — no negations
    query_parts = [f"beauty personal care products"]

    if liked_text:
        query_parts.append(liked_text[:200])

    if persona_type == "Enthusiast":
        query_parts.append("highly rated excellent quality")
    elif persona_type == "Harsh Critic":
        query_parts.append("exceptional proven effective results")
    else:
        query_parts.append("good value reliable quality")

    query = ' '.join(query_parts)[:500]

    return persona, query


# Test it on a real user with history
rich_users = df.groupby('user_id').filter(
    lambda x: len(x) >= 5
)['user_id'].unique()

test_user_id = rich_users[0]
persona, query = extract_persona_from_history(test_user_id, df)

print("USER PERSONA EXTRACTED")
print("="*60)
print(f"User ID      : {persona['user_id'][:20]}...")
print(f"Reviews      : {persona['review_count']}")
print(f"Avg Rating   : {persona['avg_rating']} (std: {persona['rating_std']})")
print(f"Persona Type : {persona['persona_type']}")
print(f"Avg Length   : {persona['avg_text_len']} words/review")
print(f"Verified     : {persona['verified_pct']*100:.0f}% purchases")
print(f"\nBehaviour samples:")
print(persona['behaviour_samples'])
print(f"\nRetrieval query: {query[:150]}")

USER PERSONA EXTRACTED
User ID      : AFSKPY37N3C43SOI5IEX...
Reviews      : 15
Avg Rating   : 4.47 (std: 0.92)
Persona Type : Enthusiast
Avg Length   : 103.7 words/review
Verified     : 0% purchases

Behaviour samples:
  Rating: 3 | "I try to get Keratin treatments every 3 months, but honestly it has been getting costly. So, when I saw this I was excit"
  Rating: 5 | "This is a really nice moisturizing lotion. It goes on lightly and is readily absorbed into my skin. My skin feels amazin"
  Rating: 3 | "I was very disappointed when I got this facial scrub. I had assumed it was like other scrubs I use but it wasn't. This i"
  Rating: 5 | "I get Keratin treatments at the salon at least 3-4 times a year (would do it more often if I could afford it). I am alwa"
  Rating: 5 | "This is perfect for my between salon visits. I have been using this now twice a week for over a month and I absolutely l"

Retrieval query: beauty personal care products I love this combo package, particularly the flo

In [36]:
def recommend_from_history(user_id, df, nigerian_mode=True, n_results=5):
    """
    Full recommendation pipeline for users WITH history.
    Counterpart to the elicitation pipeline for cold-start users.
    """
    result = extract_persona_from_history(user_id, df)

    if result is None:
        print("No history found — switching to cold-start elicitation.")
        return full_recommendation_pipeline(nigerian_mode)

    persona, query = result

    # Build human-readable persona string for LLM reranker
    persona_string = (
        f"{persona['persona_type']} with {persona['review_count']} reviews. "
        f"Gives average rating of {persona['avg_rating']}/5. "
        f"Recent behaviour:\n{persona['behaviour_samples']}"
    )

    if persona['disliked_text']:
        persona_string += (
            f"\nProducts they were disappointed by: "
            f"{persona['disliked_text'][:150]}"
        )

    print(f" Persona: {persona['persona_type']} "
          f"({persona['review_count']} reviews, "
          f"avg {persona['avg_rating']}★)\n")

    # Retrieve
    print(" Retrieving candidates from history-based persona...")
    candidates = retrieve_items(query, n_results=20)

    # Filter out items user already reviewed
    seen_asins = set(
        df[df['user_id'] == user_id]['parent_asin']
    )
    candidates = [
        c for c in candidates
        if c['asin'] not in seen_asins
    ][:10]

    print(f"   {len(candidates)} unseen candidates found\n")

    # LLM rerank
    print("🤖 LLM reranking with persona context...")
    rerank_result = llm_rerank_safe(persona_string, candidates, nigerian_mode)

    # Display
    print("=" * 60)
    print("PERSONALISED RECOMMENDATIONS (History-Based)")
    print("=" * 60)
    print(f"\n {rerank_result['reasoning']}\n")

    for i, item in enumerate(rerank_result['reranked'][:n_results], 1):
        print(f"{i}. {item['title'][:65]}")
        print(f"   ⭐ {item['avg_rating']} | {item['review_count']} reviews")
        print(f"    {item.get('explanation', '')}")
        print()

    return rerank_result, persona


# Test it
output, persona = recommend_from_history(test_user_id, df, nigerian_mode=True)

 Persona: Enthusiast (15 reviews, avg 4.47★)

 Retrieving candidates from history-based persona...
   10 unseen candidates found

🤖 LLM reranking with persona context...
PERSONALISED RECOMMENDATIONS (History-Based)

 This user values practical, effective solutions and is very conscious about getting good value for money, as shown by their concerns about costly treatments. They appreciate products that deliver on their promises and are reliable, often giving high ratings to items that are "perfect" or "amazing." The recommendations prioritize high ratings, proven effectiveness, and clear benefits, keeping the Nigerian consumer's focus on practicality and smart spending in mind.

1. As an emergency "just ate a steak at a restaurant and need floss 
   ⭐ 4.7 | 102 reviews
    This one na lifesaver for small small emergencies, e go save you when you need am quick, no time to waste.

2. Repeat Buyer - Toothbrushes are Great
   ⭐ 4.63 | 35 reviews
    If people dey buy am again and again, you

In [37]:
# Cell 28 — Unified entry point (0 reviews / 1 review / 2+ reviews)
def recommend(user_id=None, nigerian_mode=True, language='pidgin'):
    if user_id is not None:
        user_history = df[df['user_id'] == user_id]
        count        = len(user_history)

        if count >= 2:
            print(f" Returning user — {count} reviews.")
            return recommend_from_history(user_id, df, nigerian_mode)

        elif count == 1:
            print(" 1-review user — extracting 1-shot persona...")
            result = extract_oneshot_persona(user_id, df)
            if result:
                persona, query, persona_string = result
                print(f"   {persona['persona_type']} | {persona['beauty_focus']}\n")
                candidates = retrieve_items(query, n_results=20)
                seen       = set(user_history['parent_asin'])
                filtered   = [c for c in candidates if c['asin'] not in seen][:10]
                reranked   = llm_rerank_safe(persona_string, filtered, nigerian_mode)
                print("="*60)
                print("RECOMMENDATIONS (1-Shot Persona)")
                print("="*60)
                print(f"\n {reranked['reasoning']}\n")
                for i, item in enumerate(reranked['reranked'][:5], 1):
                    print(f"{i}. {item['title'][:65]}")
                    print(f"   ⭐ {item['avg_rating']} | {item['review_count']} reviews")
                    print(f"   💬 {item.get('explanation', '')}")
                    print()
                return reranked, persona
            else:
                return recommend_multilingual(language, nigerian_mode)
        else:
            print(" New user — elicitation.\n")
            return recommend_multilingual(language, nigerian_mode)
    else:
        print("No user ID — elicitation.\n")
        return recommend_multilingual(language, nigerian_mode)

print("recommend() unified entry point loaded")

recommend() unified entry point loaded


In [38]:
# Cell A — 1-Shot Persona Extraction
def extract_oneshot_persona(user_id, df):
    user_reviews = df[df['user_id'] == user_id]
    if len(user_reviews) == 0:
        return None

    review = user_reviews.iloc[0]

    prompt = f"""
A user left this single Amazon Beauty review:

Rating: {int(review['rating'])}/5
Title: "{review['title']}"
Review: "{str(review['text'])[:300]}"

Extract a concise persona. Output EXACTLY:

PERSONA_TYPE: [Enthusiast/Balanced/Critic]
VALUES: [what they care about, 1 sentence]
DISLIKES: [what they dislike, 1 sentence]
STYLE: [verbose/brief/emotional/analytical]
BEAUTY_FOCUS: [skincare/haircare/makeup/fragrance/general]
""".strip()

    try:
        raw = generate(prompt)

        def extract_field(field, text):
            match = re.search(
                rf'{field}:\s*(.+?)(?=\n[A-Z_]+:|$)',
                text, re.DOTALL
            )
            return match.group(1).strip() if match else ""

        persona = {
            'user_id'      : user_id,
            'source'       : 'oneshot',
            'rating'       : int(review['rating']),
            'persona_type' : extract_field('PERSONA_TYPE', raw),
            'values'       : extract_field('VALUES', raw),
            'dislikes'     : extract_field('DISLIKES', raw),
            'style'        : extract_field('STYLE', raw),
            'beauty_focus' : extract_field('BEAUTY_FOCUS', raw),
        }

        query          = (
            f"{persona['beauty_focus']} products "
            f"natural effective gentle quality"
        )
        persona_string = (
            f"{persona['persona_type']} beauty shopper. "
            f"Values: {persona['values']}. "
            f"Dislikes: {persona['dislikes']}."
        )

        return persona, query, persona_string

    except Exception as e:
        print(f"1-shot extraction failed: {e}")
        return None


# Test it
oneshot_users = df.groupby('user_id').filter(
    lambda x: len(x) == 1
)['user_id'].unique()

result = extract_oneshot_persona(oneshot_users[0], df)
if result:
    persona, query, persona_string = result
    print("1-SHOT PERSONA")
    print("="*50)
    for k, v in persona.items():
        if k not in ['user_id', 'source']:
            print(f"{k:<15}: {v}")
    print(f"\nQuery  : {query}")
    print(f"Persona: {persona_string}")

1-SHOT PERSONA
rating         : 5
persona_type   : Enthusiast
values         : They value products that deliver immediate and positive sensory experiences, specifically a pleasant scent and a great feel.
dislikes       : No dislikes are expressed, indicating high satisfaction with the product's sensory and performance attributes.
style          : brief/emotional
beauty_focus   : general

Query  : general products natural effective gentle quality
Persona: Enthusiast beauty shopper. Values: They value products that deliver immediate and positive sensory experiences, specifically a pleasant scent and a great feel.. Dislikes: No dislikes are expressed, indicating high satisfaction with the product's sensory and performance attributes..


In [39]:
# Cell B — Multilingual Nigerian Prompts
LANGUAGE_PROMPTS = {
    'pidgin': {
        'name'    : 'Nigerian Pidgin',
        'greeting': "Wetin you need today?",
        'system'  : """
You are recommending products to a Nigerian Pidgin English speaker.
Respond naturally in Pidgin mixed with English — like a trusted friend.
Example: "This one e good o! E no go break your pocket.
Many people don use am, dem all happy. I go advise make you try am!"
"""
    },
    'yoruba': {
        'name'    : 'Yoruba-English',
        'greeting': "Ẹ káàbọ̀! How can I help you?",
        'system'  : """
You are recommending products to a Yoruba speaker.
Mix Yoruba phrases naturally with English.
Example: "Ẹ káàbọ̀! This product o dára gan-an.
Ó tọ́ iye rẹ̀ — worth every naira. Ẹ gbìyànjú!"
"""
    },
    'hausa': {
        'name'    : 'Hausa-English',
        'greeting': "Sannu! How can I help you?",
        'system'  : """
You are recommending products to a Hausa speaker.
Mix Hausa phrases naturally with English.
Example: "Sannu! Wannan kaya yana da kyau sosai.
Mai rahusa ne — affordable and effective. Ina ba da shawara!"
"""
    },
    'igbo': {
        'name'    : 'Igbo-English',
        'greeting': "Nnọọ! How can I help you?",
        'system'  : """
You are recommending products to an Igbo speaker.
Mix Igbo phrases naturally with English.
Example: "Nnọọ! Ihe a dị mma nke ọma.
O dị ire ire — good value. Nwanne, I ga-amasị ya!"
"""
    },
    'english': {
        'name'    : 'Nigerian English',
        'greeting': "Hello! How can I help you today?",
        'system'  : """
You are recommending products to a Nigerian user.
Use Nigerian English — practical, value-conscious, communal framing.
Reference everyday Nigerian consumer concerns naturally.
"""
    }
}

print("Language prompts loaded:", list(LANGUAGE_PROMPTS.keys()))

Language prompts loaded: ['pidgin', 'yoruba', 'hausa', 'igbo', 'english']


In [40]:
# Cell C — Multilingual Recommendation Pipeline
def recommend_multilingual(language='pidgin', nigerian_mode=True):
    lang   = LANGUAGE_PROMPTS.get(language, LANGUAGE_PROMPTS['english'])
    print("="*60)
    print(f"RECOMMENDATION AGENT — {lang['name'].upper()}")
    print("="*60)
    print(f"{lang['greeting']}\n")

    answers   = {}
    questions = [
        ("q1", "What beauty products do you use most?",
               "e.g. skincare, haircare, fragrance, makeup"),
        ("q2", "What matters most when buying?",
               "e.g. price, natural ingredients, effectiveness"),
        ("q3", "Anything you avoid?",
               "e.g. alcohol, strong fragrances, sulphates")
    ]

    for qid, question, example in questions:
        print(f"Q: {question}")
        print(f"   ({example})")
        answers[qid] = input("Your answer: ").strip()
        print()

    persona_query = (
        f"User who uses {answers['q1']} products. "
        f"Prioritises {answers['q2']}. "
        f"Avoids {answers['q3']}."
    )
    query, avoid = build_retrieval_query(answers)

    print("🔍 Finding products...")
    candidates = retrieve_items(query, n_results=20)
    filtered   = post_filter(candidates, avoid)[:10]
    if not filtered:
        filtered = candidates[:10]

    # Build language-aware reranker prompt
    import random
    shuffled = filtered.copy()
    random.shuffle(shuffled)

    candidate_list = "\n".join([
        f"{i+1}. {item['title'][:80]} "
        f"(⭐{item['avg_rating']:.1f}, {item['review_count']} reviews)"
        for i, item in enumerate(shuffled)
    ])

    prompt = f"""
{lang['system']}

USER PERSONA: {persona_query}

CANDIDATE PRODUCTS:
{candidate_list}

Rerank these for this user. Respond in {lang['name']}.

Output EXACTLY:

REASONING: [2-3 sentences in {lang['name']}]
RANKING:
1. [title] | [one sentence in {lang['name']}]
2. [title] | [one sentence in {lang['name']}]
3. [title] | [one sentence in {lang['name']}]
4. [title] | [one sentence in {lang['name']}]
5. [title] | [one sentence in {lang['name']}]
""".strip()

    print("🤖 Personalising in", lang['name'], "...")

    try:
        raw             = generate(prompt)
        reasoning_match = re.search(
            r'REASONING:\s*(.*?)(?=RANKING:)', raw, re.DOTALL
        )
        reasoning       = reasoning_match.group(1).strip() if reasoning_match else ""
        ranking         = re.findall(r'\d+\.\s+(.+?)\s*\|\s*(.+)', raw)

        reranked = []
        used     = set()
        for title, explanation in ranking:
            for i, c in enumerate(shuffled):
                if i in used:
                    continue
                overlap = len(
                    set(title.lower().split()) &
                    set(c['title'].lower().split())
                )
                if overlap > 0:
                    used.add(i)
                    reranked.append({**c, 'explanation': explanation.strip()})
                    break

        if not reranked:
            reranked = filtered[:5]

    except Exception as e:
        print(f"LLM error: {e}")
        reranked  = filtered[:5]
        reasoning = "Top picks based on your preferences."

    print("\n" + "="*60)
    print(f"YOUR PICKS — {lang['name'].upper()}")
    print("="*60)
    print(f"\n💭 {reasoning}\n")

    for i, item in enumerate(reranked[:5], 1):
        print(f"{i}. {item['title'][:65]}")
        print(f"   ⭐ {item['avg_rating']} | {item['review_count']} reviews")
        print(f"   💬 {item.get('explanation', '')}")
        print()

    return reranked, language

print("recommend_multilingual() loaded")

recommend_multilingual() loaded


In [41]:
# Cell D — Test Pidgin
results, lang = recommend_multilingual(language='pidgin')

RECOMMENDATION AGENT — NIGERIAN PIDGIN
Wetin you need today?

Q: What beauty products do you use most?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  skincare



Q: What matters most when buying?
   (e.g. price, natural ingredients, effectiveness)


Your answer:  price



Q: Anything you avoid?
   (e.g. alcohol, strong fragrances, sulphates)


Your answer:  alcohol



🔍 Finding products...
🤖 Personalising in Nigerian Pidgin ...

YOUR PICKS — NIGERIAN PIDGIN

💭 My guy, I sabi say your money and your skin health important well-well. So I look for products wey no go break your pocket and at the same time, no get plenty harsh chemicals like alcohol wey fit vex your skin. I focus on those wey people don talk say dem gentle and affordable.

1. This is A Wonderful Product… it’s getting very hard to find. I Lo
   ⭐ 5.0 | 10 reviews
   💬 This one na number one because dem talk say e get "cheaper cost" and Caress na brand wey plenty people sabi, e sure say e no go cost too much.

2. Gentle Cleanser that smells amazing!
   ⭐ 4.33 | 6 reviews
   💬 This cream go fit your skin well because dem talk say e no dey break out skin, meaning no harsh chemicals, and plenty people don use am so e fit dey affordable.

3. cruelty-free face cream that doesn't break me out but still makes
   ⭐ 4.23 | 65 reviews
   💬 If your skin dey sensitive, this one good well-well because

In [42]:
# Cell E — Test Yoruba
results, lang = recommend_multilingual(language='yoruba')

RECOMMENDATION AGENT — YORUBA-ENGLISH
Ẹ káàbọ̀! How can I help you?

Q: What beauty products do you use most?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  haircare



Q: What matters most when buying?
   (e.g. price, natural ingredients, effectiveness)


Your answer:  price



Q: Anything you avoid?
   (e.g. alcohol, strong fragrances, sulphates)


Your answer:  alcohol



🔍 Finding products...
🤖 Personalising in Yoruba-English ...

YOUR PICKS — YORUBA-ENGLISH

💭 A mọ pé o fẹran awọn ọja tí kò ní ọti-lile (alcohol), ati pe o tún fẹran awọn ti o jẹ́ economical. Nitorinaa, a ti yan awọn ọja to dára jùlọ tí ó jẹ́ natural, tí yoo sì fun ọ ní value fun owo rẹ gan-an.

1. Cleans and conditions hair, soft, smooth and natural! Great
   ⭐ 4.27 | 15 reviews
   💬 Ọja yìí jẹ́ gentle gan-an, ó sì natural patapata, kò ní ohun tí yóò pa irun rẹ lára.

2. Best scented hair spray i have used.
   ⭐ 5.0 | 12 reviews
   💬 Pẹ̀lú àwọn natural products tí wọ́n fi ṣe ọja yìí, irun rẹ yoo lágbára, yóò sì dán yín lójú.

3. It is gentle and totally natural
   ⭐ 5.0 | 5 reviews
   💬 Ó máa ń wẹ, ó sì máa ń conditioner irun rẹ lọ́kọ̀ọ̀kan, ó jẹ́ natural pẹ̀lú, nítorí náà ó dára fún iye owó rẹ̀.

4. although the fragrance was wonderful.
   ⭐ 4.67 | 3 reviews
   💬 Shampoo àti conditioner yìí dára gan-an, kò sì ní fragrance, èyí tó dára fún àwọn tó n yẹra fún àwọn ohun tó ní odor.

5. 

In [43]:
# Cell F — Test Hausa
results, lang = recommend_multilingual(language='hausa')

RECOMMENDATION AGENT — HAUSA-ENGLISH
Sannu! How can I help you?

Q: What beauty products do you use most?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  skincare



Q: What matters most when buying?
   (e.g. price, natural ingredients, effectiveness)


Your answer:  effectiveness



Q: Anything you avoid?
   (e.g. alcohol, strong fragrances, sulphates)


Your answer:  alcohol



🔍 Finding products...
🤖 Personalising in Hausa-English ...

YOUR PICKS — HAUSA-ENGLISH

💭 Na zabi waɗannan samfurori domin sun dace da bukatunka na samun effective skincare, especially for those who prioritize gentle formulations. Kusan dukkaninsu suna ambaton cewa suna da gentle, soothing ne, ko kuma perfect for sensitive skin, wanda ke nuna they are likely alcohol-free and won't cause irritation. Bugu da ƙari, these products have impressive ratings and positive reviews, which confirms their effectiveness.

1. cruelty-free face cream that doesn't break me out but still makes
   ⭐ 4.23 | 65 reviews
   💬 Wannan product ɗin yana da kyau sosai, musamman idan fatarka dry ce ko kuma sensitive, kuma yana da cikakken 5.0-star rating.

2. The BEST skin care product you can buy
   ⭐ 5.0 | 3 reviews
   💬 Wannan product yana da matuƙar gentle a jiki, kuma zai sa fatarka ta yi laushi sosai, yana da 5.0-star rating shima.

3. Soothing cleanse results in silky skin
   ⭐ 5.0 | 3 reviews
   💬 Wannan 

In [44]:
# Cell G — Reranker NDCG Evaluation (30 users)
def evaluate_with_vs_without_reranker(df, item_meta, embeddings, n_users=30):

    asin_to_idx = {
        row['parent_asin']: idx
        for idx, row in item_meta.iterrows()
    }

    user_counts = df.groupby('user_id').size()
    eligible    = user_counts[user_counts >= 2].index.tolist()
    test_users  = random.sample(eligible, min(n_users, len(eligible)))

    ndcg_blend    = []
    hit_blend     = []
    ndcg_reranked = []
    hit_reranked  = []
    skipped       = 0

    print(f"Evaluating {len(test_users)} users (blend vs reranker)...\n")

    for i, user_id in enumerate(test_users):
        user_history  = df[df['user_id'] == user_id].sort_values('timestamp')
        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx:
            skipped += 1
            continue

        unseen = [
            a for a in list(asin_to_idx.keys())
            if a not in seen_asins
        ]
        if len(unseen) < 99:
            skipped += 1
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        history    = user_history.iloc[:-1]
        avg_r      = history['rating'].mean() if len(history) > 0 else 3.96
        liked_text = ' '.join(
            history[history['rating'] >= 4]['text'].fillna('').tolist()[:3]
        )[:300]
        persona    = (
            f"Beauty shopper. Avg rating: {avg_r:.1f}/5. "
            f"Liked: {liked_text}"
        )

        pool_indices    = [asin_to_idx[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        query_vec = embedder.encode([persona]).astype('float32')
        faiss.normalize_L2(query_vec)

        sem_scores   = np.dot(pool_embeddings, query_vec.T).flatten()
        pop_scores   = np.array([
            pop_lookup.get(a, 0.0) for a in candidate_pool
        ])
        blend_scores  = 0.5 * sem_scores + 0.5 * pop_scores
        blend_order   = np.argsort(blend_scores)[::-1]
        blend_asins   = [candidate_pool[j] for j in blend_order]

        # Blend only
        ndcg_blend.append(compute_ndcg_at_k(blend_asins, positive_asin))
        hit_blend.append(hit_rate_at_k(blend_asins, positive_asin))

        # Blend + LLM reranker
        try:
            top10 = []
            for a in blend_asins[:10]:
                if a in asin_to_idx:
                    m = item_meta.iloc[asin_to_idx[a]]
                    top10.append({
                        'asin'        : a,
                        'title'       : str(m['title']),
                        'avg_rating'  : float(m['avg_rating']),
                        'review_count': int(m['review_count']),
                    })

            result         = llm_rerank_safe(persona, top10)
            reranked_asins = [item['asin'] for item in result['reranked']]
            remaining      = [a for a in blend_asins if a not in reranked_asins]
            final_asins    = reranked_asins + remaining

            ndcg_reranked.append(compute_ndcg_at_k(final_asins, positive_asin))
            hit_reranked.append(hit_rate_at_k(final_asins, positive_asin))
            time.sleep(1)

        except Exception:
            ndcg_reranked.append(ndcg_blend[-1])
            hit_reranked.append(hit_blend[-1])

        if (i + 1) % 5 == 0:
            print(
                f"  {i+1}/{len(test_users)} | "
                f"Blend: {np.mean(ndcg_blend):.4f}/{np.mean(hit_blend):.4f} | "
                f"Reranked: {np.mean(ndcg_reranked):.4f}/{np.mean(hit_reranked):.4f}"
            )

    print("\n" + "="*60)
    print("RERANKER IMPACT")
    print("="*60)
    print(f"{'Strategy':<30} {'NDCG@10':>8} {'Hit@10':>8}")
    print("-"*60)
    print(f"{'Blend only':<30} {np.mean(ndcg_blend):>8.4f} {np.mean(hit_blend):>8.4f}")
    print(f"{'Blend + LLM reranker':<30} {np.mean(ndcg_reranked):>8.4f} {np.mean(hit_reranked):>8.4f}")
    print("="*60)
    delta = np.mean(ndcg_reranked) - np.mean(ndcg_blend)
    print(f"\nReranker NDCG delta: {delta:+.4f}")

    return np.mean(ndcg_reranked), np.mean(hit_reranked)

ndcg_final, hit_final = evaluate_with_vs_without_reranker(
    df, item_meta, embeddings, n_users=30
)

Evaluating 30 users (blend vs reranker)...

  5/30 | Blend: 0.6528/1.0000 | Reranked: 0.5071/1.0000
  10/30 | Blend: 0.4384/0.8000 | Reranked: 0.3712/0.8000
  15/30 | Blend: 0.4058/0.7333 | Reranked: 0.3762/0.7333
  20/30 | Blend: 0.3732/0.7000 | Reranked: 0.3510/0.7000
  25/30 | Blend: 0.3632/0.6800 | Reranked: 0.3455/0.6800
  30/30 | Blend: 0.3448/0.6667 | Reranked: 0.3300/0.6667

RERANKER IMPACT
Strategy                        NDCG@10   Hit@10
------------------------------------------------------------
Blend only                       0.3448   0.6667
Blend + LLM reranker             0.3300   0.6667

Reranker NDCG delta: -0.0148


In [45]:
# Sanity check — does history-based pipeline improve NDCG
# compared to cold-start for users who DO have history?

def evaluate_history_users(df, item_meta, embeddings, n_users=30):
    """
    Evaluate specifically on users with ≥3 reviews.
    These are the users where history-based persona should shine.
    """
    asin_to_idx_local = {
        row['parent_asin']: idx
        for idx, row in item_meta.iterrows()
    }

    rich_user_ids = df.groupby('user_id').filter(
        lambda x: len(x) >= 3
    )['user_id'].unique().tolist()

    test_users = random.sample(
        rich_user_ids, min(n_users, len(rich_user_ids))
    )

    ndcg_scores = []
    hit_scores  = []

    print(f"Evaluating {len(test_users)} history users...\n")

    for user_id in test_users:
        user_history  = df[df['user_id'] == user_id].sort_values('timestamp')
        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx_local:
            continue

        unseen = [
            a for a in list(asin_to_idx_local.keys())
            if a not in seen_asins
        ]
        if len(unseen) < 99:
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        # Use history-based persona for retrieval
        persona_result = extract_persona_from_history(user_id, df)
        if persona_result is None:
            continue

        _, query = persona_result

        pool_indices    = [asin_to_idx_local[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        query_vec = embedder.encode([query]).astype('float32')
        faiss.normalize_L2(query_vec)

        sem_scores   = np.dot(pool_embeddings, query_vec.T).flatten()
        pop_scores   = np.array([
            pop_lookup.get(a, 0.0) for a in candidate_pool
        ])

        # Use confirmed best blend
        final_scores = 0.5 * sem_scores + 0.5 * pop_scores
        ranked_asins = [
            candidate_pool[j] for j in np.argsort(final_scores)[::-1]
        ]

        ndcg_scores.append(
            compute_ndcg_at_k(ranked_asins, positive_asin, k=10)
        )
        hit_scores.append(
            hit_rate_at_k(ranked_asins, positive_asin, k=10)
        )

    print("=" * 50)
    print("HISTORY-BASED USERS EVALUATION")
    print("=" * 50)
    print(f"Users evaluated : {len(ndcg_scores)}")
    print(f"NDCG@10         : {np.mean(ndcg_scores):.4f}")
    print(f"Hit Rate@10     : {np.mean(hit_scores):.4f}")
    print("=" * 50)
    print("\nFor comparison — all users (cold-start included):")
    print("NDCG@10: 0.4210 | Hit@10: 0.6600")

    return np.mean(ndcg_scores), np.mean(hit_scores)

ndcg_h, hit_h = evaluate_history_users(df, item_meta, embeddings, n_users=30)

Evaluating 30 history users...

HISTORY-BASED USERS EVALUATION
Users evaluated : 30
NDCG@10         : 0.2566
Hit Rate@10     : 0.5000

For comparison — all users (cold-start included):
NDCG@10: 0.4210 | Hit@10: 0.6600


In [ ]:
def evaluate_with_reranker(df, item_meta, embeddings, n_users=30):
    """
    Proper evaluation that includes the LLM reranker.
    Expensive (API calls) so use smaller n_users.
    """
    asin_to_idx_local = {
        row['parent_asin']: idx
        for idx, row in item_meta.iterrows()
    }

    user_counts    = df.groupby('user_id').size()
    eligible       = user_counts[user_counts >= 2].index.tolist()
    test_users     = random.sample(eligible, min(n_users, len(eligible)))

    ndcg_scores = []
    hit_scores  = []
    skipped     = 0

    print(f"Evaluating {len(test_users)} users WITH LLM reranker...\n")

    for i, user_id in enumerate(test_users):
        user_history  = df[df['user_id'] == user_id].sort_values('timestamp')
        positive_asin = user_history.iloc[-1]['parent_asin']
        seen_asins    = set(user_history['parent_asin'])

        if positive_asin not in asin_to_idx_local:
            skipped += 1
            continue

        unseen = [
            a for a in list(asin_to_idx_local.keys())
            if a not in seen_asins
        ]
        if len(unseen) < 99:
            skipped += 1
            continue

        negatives      = random.sample(unseen, 99)
        candidate_pool = negatives + [positive_asin]
        random.shuffle(candidate_pool)

        # Build persona
        history    = user_history.iloc[:-1]
        avg_r      = history['rating'].mean() if len(history) > 0 else 3.96
        liked_text = ' '.join(
            history[history['rating'] >= 4]['text'].fillna('').tolist()[:3]
        )[:300]

        persona_string = (
            f"Beauty shopper. Avg rating: {avg_r:.1f}/5. "
            f"Liked: {liked_text}"
        )

        # Step 1 — FAISS + blend (same as v2)
        pool_indices    = [asin_to_idx_local[a] for a in candidate_pool]
        pool_embeddings = embeddings[pool_indices].astype('float32')
        faiss.normalize_L2(pool_embeddings)

        query_vec = embedder.encode([persona_string]).astype('float32')
        faiss.normalize_L2(query_vec)

        sem_scores   = np.dot(pool_embeddings, query_vec.T).flatten()
        pop_scores   = np.array([
            pop_lookup.get(a, 0.0) for a in candidate_pool
        ])
        final_scores = 0.5 * sem_scores + 0.5 * pop_scores

        # Take top 10 from blend for reranker input
        top10_indices  = np.argsort(final_scores)[::-1][:10]
        top10_asins    = [candidate_pool[j] for j in top10_indices]
        top10_items    = [
            {
                'asin'        : a,
                'title'       : item_meta[
                    item_meta['parent_asin'] == a
                ]['title'].values[0] if a in asin_to_idx_local else a,
                'avg_rating'  : item_meta[
                    item_meta['parent_asin'] == a
                ]['avg_rating'].values[0] if a in asin_to_idx_local else 3.0,
                'review_count': item_meta[
                    item_meta['parent_asin'] == a
                ]['review_count'].values[0] if a in asin_to_idx_local else 1,
            }
            for a in top10_asins
        ]

        # Step 2 — LLM reranker
        try:
            rerank_result  = llm_rerank_safe(
                persona_string, top10_items, nigerian_mode=True
            )
            reranked_items = rerank_result['reranked']
            reranked_asins = [item['asin'] for item in reranked_items]

            # Fill remaining spots with blend order if reranker returned < 10
            remaining = [
                a for a in top10_asins if a not in reranked_asins
            ]
            reranked_asins = reranked_asins + remaining

            time.sleep(1)  # rate limiting

        except Exception as e:
            print(f"Reranker failed for user {i}: {e}")
            reranked_asins = top10_asins  # fallback to blend order

        ndcg = compute_ndcg_at_k(reranked_asins, positive_asin, k=10)
        hit  = hit_rate_at_k(reranked_asins, positive_asin, k=10)

        ndcg_scores.append(ndcg)
        hit_scores.append(hit)

        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{len(test_users)} | "
                  f"NDCG@10: {np.mean(ndcg_scores):.4f} | "
                  f"Hit@10: {np.mean(hit_scores):.4f}")

    print("\n" + "="*55)
    print("EVALUATION WITH LLM RERANKER")
    print("="*55)
    print(f"Users evaluated : {len(ndcg_scores)}")
    print(f"Skipped         : {skipped}")
    print(f"NDCG@10         : {np.mean(ndcg_scores):.4f}")
    print(f"Hit Rate@10     : {np.mean(hit_scores):.4f}")
    print("="*55)
    print("\nComparison:")
    print(f"  FAISS + blend only : NDCG 0.4210 | Hit 0.6600")
    print(f"  + LLM reranker     : NDCG {np.mean(ndcg_scores):.4f} "
          f"| Hit {np.mean(hit_scores):.4f}")

    return np.mean(ndcg_scores), np.mean(hit_scores)

# Run on 30 users (API calls make this slow — ~5 mins)
ndcg_r, hit_r = evaluate_with_reranker(df, item_meta, embeddings, n_users=30)

Evaluating 30 users WITH LLM reranker...

  5/30 | NDCG@10: 0.2559 | Hit@10: 0.6000
  10/30 | NDCG@10: 0.3095 | Hit@10: 0.6000
  15/30 | NDCG@10: 0.3151 | Hit@10: 0.5333
  20/30 | NDCG@10: 0.3109 | Hit@10: 0.5500
